In [1]:
print("hello world")

hello world


In [2]:
import torch
from datasets import load_dataset
from transformers import BartTokenizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Working on {device}")

Working on cuda


In [3]:
model_checkpoint = "facebook/bart-large-cnn"
tokenizer = BartTokenizer.from_pretrained(model_checkpoint)

print("Loading the CNN daily mail dataset from Hugging Face")
raw_datasets = load_dataset("cnn_dailymail","3.0.0")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loading the CNN daily mail dataset from Hugging Face


...

3.0.0/validation-00000-of-00001.parquet:   0%|          | 0.00/34.7M [00:00<?, ?B/s]

3.0.0/test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

In [11]:
print(raw_datasets['train']['article'][-1])

A Facebook page seeking to preserve the 'Black Pete' clowns in blackface who accompany St. Nicholas to the Netherlands during the holidays has become the fastest-growing Dutch-language page ever, receiving 1 million 'likes' in a single day. The popularity of the 'Pete-ition' page reflects the emotional attachment most Dutch have to a figure that helped launch the tradition of Santa Claus. It also reflects their anger at critics who call it racist. Those critics include foreigners who they feel don't understand the tradition. They also include many of the country's most prominent black people. Facebook page: The U.N. has condemned the Christmas tradition but a Facebook page set up in support of it has attracted huge attention . 'Don't let the Netherlands' most beautiful tradition disappear,' the page says. On Tuesday, the chairwoman of a U.N. Human Rights Commission panel condemned it. 'The working group does not understand why it is that people in the Netherlands cannot see that this i

In [14]:
print(raw_datasets['train'].features)
print(raw_datasets['test'].features)
print(raw_datasets['validation'].features)

{'article': Value('string'), 'highlights': Value('string'), 'id': Value('string')}
{'article': Value('string'), 'highlights': Value('string'), 'id': Value('string')}
{'article': Value('string'), 'highlights': Value('string'), 'id': Value('string')}


In [16]:
print(raw_datasets['train'].info)

DatasetInfo(description='', citation='', homepage='', license='', features={'article': Value('string'), 'highlights': Value('string'), 'id': Value('string')}, post_processed=None, supervised_keys=None, builder_name='parquet', dataset_name='cnn_dailymail', config_name='3.0.0', version=0.0.0, splits={'train': SplitInfo(name='train', num_bytes=1261703785, num_examples=287113, shard_lengths=[115705, 115704, 55704], dataset_name='cnn_dailymail'), 'validation': SplitInfo(name='validation', num_bytes=57732412, num_examples=13368, shard_lengths=None, dataset_name='cnn_dailymail'), 'test': SplitInfo(name='test', num_bytes=49925732, num_examples=11490, shard_lengths=None, dataset_name='cnn_dailymail')}, download_checksums={'hf://datasets/cnn_dailymail@96df5e686bee6baa90b8bee7c28b81fa3fa6223d/3.0.0/train-00000-of-00003.parquet': {'num_bytes': 256540614, 'checksum': None}, 'hf://datasets/cnn_dailymail@96df5e686bee6baa90b8bee7c28b81fa3fa6223d/3.0.0/train-00001-of-00003.parquet': {'num_bytes': 25658

In [18]:
print(raw_datasets['train'].info.builder_name)

parquet


In [5]:
# Check a random row
print(f"Article: {raw_datasets['train'][0]['article'][:500]}...")
print(f"\nSummary: {raw_datasets['train'][0]['highlights']}")

Article: LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won't cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash away on fast cars, drink and celebrity parties. "I don't plan to be one of those people who, as s...

Summary: Harry Potter star Daniel Radcliffe gets £20M fortune as he turns 18 Monday .
Young actor says he has no plans to fritter his cash away .
Radcliffe's earnings from first five Potter films have been held in trust fund .


In [ ]:
# MODEL DEFINITION
import torch.nn as nn
from transformers import BartModel

class Summarization_head(nn.Module):
    def __init__(self, input_dim, vocab_size):
        super().__init__()
        self.custom_layers = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(512, vocab_size)
        )

    def forward(self, x):
        return self.custom_layers(x)

class MyBartSummarizer(nn.Module):
    def __init__(self, checkpoint="facebook/bart-base"):
        super().__init__()
        self.bart = BartModel.from_pretrained(checkpoint)
        
        # Freeze BART parameters
        for param in self.bart.parameters():
            param.requires_grad = False
        
        self.head = Summarization_head(input_dim=768, vocab_size=50265)

    def forward(self, input_ids, attention_mask, decoder_input_ids):
        outputs = self.bart(
            input_ids=input_ids,
            attention_mask=attention_mask,
            decoder_input_ids=decoder_input_ids
        )
        logits = self.head(outputs.last_hidden_state)
        return logits

print("Model classes loaded!")

In [ ]:
# TRAINING FUNCTION
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

def prepare_data():
    def tokenize_fn(batch):
        # Tokenize input articles
        inputs = tokenizer(batch["article"], truncation=True, padding="max_length", max_length=512)
        # Tokenize target summaries
        with tokenizer.as_target_tokenizer():
            labels = tokenizer(batch["highlights"], truncation=True, padding="max_length", max_length=128)
        inputs["labels"] = labels["input_ids"]
        return inputs
    
    # Use subset for faster training
    tokenized_dataset = raw_datasets["train"].select(range(1000)).map(tokenize_fn, batched=True)
    tokenized_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
    return DataLoader(tokenized_dataset, batch_size=4, shuffle=True)

def train():
    print(f"Device: {device}")
    dataloader = prepare_data()
    model = MyBartSummarizer(model_checkpoint).to(device)
    
    # Optimizer only for the head (BART is frozen)
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()), 
        lr=1e-4
    )
    
    loss_fct = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)
    
    model.train()
    for epoch in range(3):
        total_loss = 0
        for batch in tqdm(dataloader, desc=f"Epoch {epoch+1}/3"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            logits = model(input_ids, attention_mask, labels)
            loss = loss_fct(logits.view(-1, 50265), labels.view(-1))
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        avg_loss = total_loss / len(dataloader)
        print(f"Epoch {epoch+1}/3 - Loss: {avg_loss:.4f}")

# Start training
print("Starting training on T4 GPU...")
train()
print("Training complete!")